# VisionAssist Phase 10 — Task-balanced 10,000-example pilot

This notebook runs the Phase 10 pilot from the untouched Qwen2.5-VL base model. It restores prepared data, audits the deterministic task-quota selection, validates one real batch, trains with Drive-backed checkpoints, and evaluates the best adapter on validation and the complete frozen benchmark. Run cells in order in a fresh A100 GPU runtime.

In [ ]:
#@title 1. Settings — run before importing Torch
import os
from pathlib import Path

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.8"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
REPO_URL = "https://github.com/moshiur00/visionassist-industrial-visual-inspection.git"
REPO_BRANCH = "main"
PROJECT_ROOT = Path("/content/visionassist-industrial-visual-inspection")
DRIVE_ROOT = Path("/content/drive/MyDrive/visionassist")
DRIVE_DATA_ARCHIVE = DRIVE_ROOT / "data/visionassist_prepared_data.tar.gz"
PILOT_RUN_ID = "qwen25vl3b_qlora_pilot_v1"
PILOT_CONFIG = PROJECT_ROOT / "configs/training/qwen25vl3b_qlora_pilot.yaml"


In [ ]:
#@title 2. Mount Drive and prepare directories
from google.colab import drive
drive.mount("/content/drive")
for path in (DRIVE_ROOT / "data", DRIVE_ROOT / "checkpoints", DRIVE_ROOT / "outputs"):
    path.mkdir(parents=True, exist_ok=True)
assert DRIVE_DATA_ARCHIVE.is_file(), f"Missing: {DRIVE_DATA_ARCHIVE}"


In [ ]:
#@title 3. Clone/update repository and install dependencies
import shutil, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "uv"], check=True)
if (PROJECT_ROOT / ".git").is_dir():
    subprocess.run(["git", "fetch", "origin", REPO_BRANCH], cwd=PROJECT_ROOT, check=True)
    subprocess.run(["git", "checkout", REPO_BRANCH], cwd=PROJECT_ROOT, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", REPO_BRANCH], cwd=PROJECT_ROOT, check=True)
else:
    if PROJECT_ROOT.exists(): shutil.rmtree(PROJECT_ROOT)
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(PROJECT_ROOT)], check=True)
subprocess.run(["uv", "sync", "--extra", "training", "--extra", "dev"], cwd=PROJECT_ROOT, check=True)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip())


In [ ]:
#@title 4. Verify GPU
import torch
assert torch.cuda.is_available(), "Select a GPU runtime."
properties = torch.cuda.get_device_properties(0)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(properties.total_memory / 1024**3, 2))
print("BF16:", torch.cuda.is_bf16_supported())
subprocess.run(["nvidia-smi"], check=False)


## A. Restore and normalize prepared data

In [ ]:
#@title 5. Restore prepared data
import tarfile
required = [PROJECT_ROOT / "data/raw/visa", PROJECT_ROOT / "data/processed/visa_instructions/train.jsonl", PROJECT_ROOT / "data/processed/visa_instructions/validation.jsonl", PROJECT_ROOT / "data/benchmarks/visa_baseline_v1/benchmark.jsonl"]
if not all(path.exists() for path in required):
    with tarfile.open(DRIVE_DATA_ARCHIVE, "r:gz") as archive:
        archive.extractall(PROJECT_ROOT, filter="data")
assert all(path.exists() for path in required)
print("Prepared data ready.")


In [ ]:
#@title 6. Normalize instruction and benchmark image paths
import hashlib, json
from pathlib import PurePosixPath
MARKER = ("data", "raw", "visa")

def normalize_image_path(value):
    parts = PurePosixPath(str(value).replace("\\", "/")).parts
    lowered = tuple(part.lower() for part in parts)
    for index in range(len(parts) - len(MARKER) + 1):
        if lowered[index:index + len(MARKER)] == MARKER:
            return PurePosixPath(*parts[index:]).as_posix()
    candidate = PurePosixPath(*parts)
    if not candidate.is_absolute() and ".." not in candidate.parts: return candidate.as_posix()
    raise ValueError(value)

def normalize_jsonl(path):
    rows, changed = [], 0
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip(): continue
        row = json.loads(line)
        for message in row.get("messages", []):
            if message.get("role") != "user": continue
            for item in message.get("content", []):
                if item.get("type") == "image":
                    normalized = normalize_image_path(item["image"])
                    changed += int(normalized != item["image"])
                    item["image"] = normalized
        rows.append(row)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text("".join(json.dumps(row, ensure_ascii=False) + "\n" for row in rows), encoding="utf-8")
    temporary.replace(path)
    return len(rows), changed

instruction_root = PROJECT_ROOT / "data/processed/visa_instructions"
for split in ("train", "validation", "test"):
    print(split, normalize_jsonl(instruction_root / f"{split}.jsonl"))
benchmark = PROJECT_ROOT / "data/benchmarks/visa_baseline_v1/benchmark.jsonl"
print("benchmark", normalize_jsonl(benchmark))
benchmark_hash = hashlib.sha256(benchmark.read_bytes()).hexdigest()
manifest_path = benchmark.parent / "benchmark_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
manifest["benchmark_sha256"] = benchmark_hash
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
(benchmark.parent / "benchmark_sha256.txt").write_text(benchmark_hash + "\n", encoding="utf-8")


In [ ]:
#@title 7. Run Phase 10 tests
subprocess.run(["uv", "run", "pytest", "tests/test_phase10_sampling.py", "tests/test_phase8_training.py", "tests/test_phase7c_inference.py"], cwd=PROJECT_ROOT, check=True)


## B. Audit and configure the pilot

In [ ]:
#@title 8. Configure persistent checkpoints and audit the exact selection
import yaml
pilot = yaml.safe_load(PILOT_CONFIG.read_text(encoding="utf-8"))
pilot["checkpoints"]["persistent_output_dir"] = str(DRIVE_ROOT / "checkpoints" / PILOT_RUN_ID)
pilot["checkpoints"]["sync_every_save"] = True
PILOT_CONFIG.write_text(yaml.safe_dump(pilot, sort_keys=False), encoding="utf-8")
subprocess.run(["uv", "run", "visionassist", "training-data-audit", "--config", str(PILOT_CONFIG)], cwd=PROJECT_ROOT, check=True)
audit_path = PROJECT_ROOT / "outputs/training" / PILOT_RUN_ID / "dataset_selection_audit.json"
audit = json.loads(audit_path.read_text(encoding="utf-8"))
print(json.dumps(audit, indent=2))
assert audit["train"]["records"] == 10000
assert audit["train"]["unique_instruction_ids"] == 10000
assert audit["train"]["task_families"] == pilot["data"]["train_task_quotas"]
assert len(audit["train"]["categories"]) == 12


In [ ]:
#@title 9. Confirm checkpoint state and inspect environment
drive_checkpoint_root = DRIVE_ROOT / "checkpoints" / PILOT_RUN_ID
existing = sorted(drive_checkpoint_root.glob("checkpoint-*")) if drive_checkpoint_root.is_dir() else []
print("Existing pilot checkpoints:", [path.name for path in existing])
subprocess.run(["uv", "run", "visionassist", "training-environment", "--config", str(PILOT_CONFIG)], cwd=PROJECT_ROOT, check=True)


In [ ]:
#@title 10. One-batch forward/backward pilot smoke test
subprocess.run(["uv", "run", "visionassist", "training-smoke-test", "--config", str(PILOT_CONFIG)], cwd=PROJECT_ROOT, check=True)
report_path = PROJECT_ROOT / "outputs/training" / PILOT_RUN_ID / "one_batch_smoke_test.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print(json.dumps(report, indent=2))
assert report["passed"] and report["finite_gradients"] and report["nonzero_gradients"]


## C. Explicit pilot launch gate

Review the full quota audit, existing checkpoint list, hardware report, and one-batch result. For a new run the checkpoint list must be empty. For a resumed run it must contain only this pilot run ID.

In [ ]:
#@title 11. Start or resume the pilot
START_PILOT = False
if not START_PILOT:
    raise RuntimeError("Pilot gate is closed. Review Cells 8–10 first.")
subprocess.run(["uv", "run", "visionassist", "train-qlora", "--config", str(PILOT_CONFIG), "--resume", "latest"], cwd=PROJECT_ROOT, check=True)


In [ ]:
#@title 12. Verify and persist completed pilot
pilot_run = PROJECT_ROOT / "outputs/training" / PILOT_RUN_ID
run_manifest = json.loads((pilot_run / "run_manifest.json").read_text(encoding="utf-8"))
print(json.dumps(run_manifest, indent=2))
assert run_manifest["status"] == "completed"
assert (pilot_run / "final_adapter/adapter_model.safetensors").is_file()
drive_pilot = DRIVE_ROOT / "outputs/training" / PILOT_RUN_ID
shutil.copytree(pilot_run, drive_pilot, dirs_exist_ok=True)
print("Saved pilot artifacts to:", drive_pilot)


## D. Evaluate the best pilot adapter

In [ ]:
#@title 13. Create Drive-resumable validation and benchmark configs
base_dir = PROJECT_ROOT / "configs/inference"
evaluation_configs = {}
for split_name, source_name, limit, seed in (
    ("validation", "qwen25vl3b_overfit_checkpoint50_validation.yaml", 1000, 43),
    ("test", "qwen25vl3b_overfit_checkpoint50_test.yaml", None, 44),
):
    config = yaml.safe_load((base_dir / source_name).read_text(encoding="utf-8"))
    output = f"outputs/post_training/qwen25vl3b_pilot_best/{split_name}"
    config.update({
        "run_id": f"qwen25vl3b_pilot_best_{split_name}_v1",
        "adapter_path": "outputs/training/qwen25vl3b_qlora_pilot_v1/final_adapter",
        "output_dir": output,
        "partial_predictions_path": f"{output}/predictions.partial.jsonl",
        "predictions_path": f"{output}/predictions.jsonl",
        "errors_path": f"{output}/inference_errors.jsonl",
        "run_manifest_path": f"{output}/run_manifest.json",
        "evaluation_records_path": f"{output}/evaluation_records.jsonl",
        "subset_limit": limit, "subset_seed": seed, "overwrite": False,
        "persistent_output_dir": str(DRIVE_ROOT / "inference" / f"qwen25vl3b_pilot_best_{split_name}"),
        "persistent_sync_every": 25,
    })
    if split_name == "validation": config["benchmark_manifest_path"] = "outputs/training/qwen25vl3b_qlora_pilot_v1/dataset_manifest.json"
    path = base_dir / f"qwen25vl3b_pilot_best_{split_name}.yaml"
    path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
    evaluation_configs[split_name] = path
    print(split_name, path)


In [ ]:
#@title 14. Run validation and full frozen benchmark
for split_name in ("validation", "test"):
    subprocess.run(["uv", "run", "visionassist", "evaluate-adapter", "--config", str(evaluation_configs[split_name])], cwd=PROJECT_ROOT, check=True)
    torch.cuda.empty_cache()


In [ ]:
#@title 15. Summarize and persist evaluation results
assessment = PROJECT_ROOT / "outputs/post_training/qwen25vl3b_pilot_best"
for split_name in ("validation", "test"):
    run_dir = assessment / split_name
    summary = json.loads((run_dir / "assessment_summary.json").read_text(encoding="utf-8"))
    metrics = json.loads((run_dir / "evaluation/metrics.json").read_text(encoding="utf-8"))
    print(f"\n===== {split_name.upper()} =====")
    print(json.dumps(summary, indent=2))
    print("failure tags:", metrics.get("failure_tag_counts", {}))
    for task, values in metrics.get("per_task", {}).items():
        print(task, {key: value for key, value in values.items() if key not in {"per_label", "confusion_matrix"}})
drive_assessment = DRIVE_ROOT / "outputs/post_training/qwen25vl3b_pilot_best"
shutil.copytree(assessment, drive_assessment, dirs_exist_ok=True)
print("Saved evaluation artifacts to:", drive_assessment)


## Resume guidance

After a disconnect, rerun setup, restore/normalize data, configure persistence, and reopen the appropriate gate. Training resumes from the newest pilot checkpoint. Long inference restores partial predictions from Drive every 25 attempted records. Do not start a full-data run until Phase 10 metrics are reviewed against the promotion criteria in `README_PHASE10.md`.